# nb12 -- Pathway / complex / regulatory validation
Checks whether `lasso`'s non-cognate top-predictor genes are known biological
partners of the cognate gene, using three complementary sources:

1. **STRING** -- functional/physical association, broad coverage. Pulls the
   per-evidence-channel scores (experiments, curated databases, co-expression,
   text-mining), not just the combined score, so a hit backed by hard evidence
   can be told apart from one that's mostly literature text-mining.
2. **OmniPath complexes** -- `CORUM, ComplexPortal, hu.MAP`.
3. **TRRUST** -- literature-curated transcription-factor -> target-gene
   regulatory interactions, checked in both directions.

**Model:** `lasso` -- the variant carried forward from nb11 (best cognate
rank-1, 81.6% bootstrap-stable, most reproducible across every stress test run).

**Scope:** the gray-zone proteins -- neither in the trustworthy core (cognate
rank-1 AND bootstrap-stable) nor flagged as likely artifacts (weak raw
correlation / technical gene / large rank gap despite strong raw correlation).
These are the ambiguous cases: the model's top pick isn't the literal cognate
gene, but might still be a real biological partner rather than noise.

## Environment setup

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/covid_citeseq_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

## Imports and config

In [ ]:
import sys
sys.path.insert(0, str(BASE_PATH))

import pandas as pd

from src.analysis.pathway_validation import (
    build_trustworthy_core, flag_artifacts, build_query_set,
    fetch_string_partner_cache, string_validate_pairs,
    load_omnipath_complexes, build_gene_to_complexes, complex_validate_pairs,
    load_trrust, trrust_validate_pairs,
    combined_verdict,
    OMNIPATH_COMPLEX_RESOURCES,
)

MODEL_VARIANT = 'lasso'

NB11_RESULTS_DIR = BASE_PATH / 'results' / 'mlp_variant_comparison'
RESULTS_DIR = BASE_PATH / 'results' / 'pathway_validation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MIN_BOOTSTRAP_FREQUENCY = 0.8

## Load nb11 outputs and build the query set
`cognate_ranks`, `bootstrap_rank1`, and `raw_correlation_check` are all
already saved from nb11's run for `lasso` -- no retraining needed here.

In [ ]:
cognate_ranks = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_cognate_ranks.csv')
bootstrap_rank1 = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_bootstrap_rank1.csv')
raw_corr_check = pd.read_csv(NB11_RESULTS_DIR / f'nb11_{MODEL_VARIANT}_raw_correlation_check.csv')

print(f'cognate_ranks:   {len(cognate_ranks)} proteins')
print(f'bootstrap_rank1: {len(bootstrap_rank1)} proteins')
print(f'raw_corr_check:  {len(raw_corr_check)} proteins')

In [ ]:
trustworthy_core = build_trustworthy_core(cognate_ranks, bootstrap_rank1, MIN_BOOTSTRAP_FREQUENCY)
artifact_flags = flag_artifacts(raw_corr_check)
query_set = build_query_set(cognate_ranks, trustworthy_core, artifact_flags)

trustworthy_proteins = set(trustworthy_core.loc[trustworthy_core['trustworthy'], 'protein'])
artifact_proteins = set(artifact_flags.loc[artifact_flags['likely_artifact'], 'protein'])

print(f'Trustworthy core (skipped): {len(trustworthy_proteins)}')
print(f'Artifact-flagged (excluded): {len(artifact_proteins)}')
print(f'Query set: {len(query_set)}')

trustworthy_core.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_trustworthy_core.csv', index=False)
artifact_flags.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_artifact_flags.csv', index=False)
query_set

## 1. STRING check -- combined score AND per-evidence-channel breakdown
`escore` (experimental) and `dscore` (curated databases) are hard-evidence
channels; `tscore` (text-mining) is the weakest, most inflatable one.

In [ ]:
unique_query_genes = pd.unique(query_set[['cognate_gene', 'top_predictor_gene']].values.ravel())
print(f'Fetching STRING partners for {len(unique_query_genes)} unique genes...')

string_partner_cache = fetch_string_partner_cache(unique_query_genes)
print('STRING fetch complete.')

string_results = string_validate_pairs(query_set, string_partner_cache)

print(f"STRING-supported pairs: {string_results['string_known_interaction'].sum()} / {len(string_results)}")
print(f"  ...with hard evidence (experiments or curated database): {string_results['string_hard_evidence'].sum()}")
print(f"  ...text-mining evidence only: {string_results['string_textmining_only'].sum()}")
string_results[string_results['string_known_interaction']].sort_values('string_score', ascending=False)

## 2. OmniPath complexes
`CORUM, ComplexPortal, hu.MAP` -- CORUM alone tends to have narrow curation
scope; hu.MAP adds AP-MS-based complexes rather than literature curation alone.

In [ ]:
complex_cache_path = RESULTS_DIR / f'omnipath_complexes_{OMNIPATH_COMPLEX_RESOURCES.replace(",", "_")}.tsv'
complexes = load_omnipath_complexes(OMNIPATH_COMPLEX_RESOURCES, cache_path=complex_cache_path)
print(f'Complexes loaded ({OMNIPATH_COMPLEX_RESOURCES}): {len(complexes)}')

gene_to_complexes = build_gene_to_complexes(complexes)
print(f'Genes indexed: {len(gene_to_complexes)}')

complex_results = complex_validate_pairs(query_set, gene_to_complexes)
print(f"Complex-supported pairs: {complex_results['shared_complex'].sum()} / {len(complex_results)}")
complex_results[complex_results['shared_complex']]

## 3. TRRUST -- transcription factor -> target regulatory check
Checks whether either gene in a pair is a known TF that regulates the other,
in either direction. If the download fails, get `trrust_rawdata.human.tsv`
manually from https://www.grnpedia.org/trrust/ and place it at
`RESULTS_DIR / 'trrust_rawdata.human.tsv'`.

In [ ]:
trrust_cache_path = RESULTS_DIR / 'trrust_rawdata.human.tsv'
trrust = load_trrust(cache_path=trrust_cache_path)
print(f'TRRUST interactions loaded: {len(trrust)}')

trrust_results = trrust_validate_pairs(query_set, trrust)
print(f"TRRUST-supported (regulatory) pairs: {trrust_results['trrust_regulatory_pair'].sum()} / {len(trrust_results)}")
trrust_results[trrust_results['trrust_regulatory_pair']]

## Combined verdict
A pair counts as biologically supported if STRING, the complex check, or
TRRUST confirms it. `evidence_type` distinguishes physical (complex),
regulatory (TRRUST), and general-association (STRING) support, and separates
hard evidence from text-mining-only STRING hits.

In [ ]:
combined = combined_verdict(string_results, complex_results, trrust_results)
combined = combined.merge(query_set[['protein', 'cognate_rank']], on='protein')

print(f"Total queried: {len(combined)}")
print(f"Biologically supported (any source): {combined['biologically_supported'].sum()}")
print(f"Unexplained (no source supports): {(~combined['biologically_supported']).sum()}")
print('\nBy evidence type:')
print(combined['evidence_type'].value_counts())

combined.sort_values(['biologically_supported', 'string_score'], ascending=[False, False])

## Save results

In [ ]:
combined.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_pathway_validation_results.csv', index=False)
combined[combined['biologically_supported']].to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_biologically_supported_pairs.csv', index=False)
combined[~combined['biologically_supported']].to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_unexplained_pairs.csv', index=False)

print(f'Saved to {RESULTS_DIR}')
print(f'  {MODEL_VARIANT}_trustworthy_core.csv             -- cognate rank-1 AND bootstrap-stable, {trustworthy_core["trustworthy"].sum()}/{len(trustworthy_core)}')
print(f'  {MODEL_VARIANT}_artifact_flags.csv                -- likely-artifact top picks, {artifact_flags["likely_artifact"].sum()}/{len(artifact_flags)}')
print(f'  {MODEL_VARIANT}_pathway_validation_results.csv    -- STRING + complexes + TRRUST, all queried (gray-zone) pairs')
print(f'  {MODEL_VARIANT}_biologically_supported_pairs.csv  -- confirmed by at least one source, with evidence_type')
print(f'  {MODEL_VARIANT}_unexplained_pairs.csv             -- no source supports -- literature spot-check or exclusion candidates')

## Follow-up: is the unexplained residual explained by cell type instead?
65.5% of the query set (38/58) had no support from any of the three
databases above. Two cheap checks before treating that as "the model found
nothing real":

1. **Extended technical-gene re-check.** `flag_artifacts` in nb11 only
   caught hemoglobin/mitochondrial/ribosomal genes. Broader ambient markers
   (actin, tubulin, GAPDH, thymosin beta, ferritin, translation elongation
   factors) weren't in that list -- cheap to check whether any of the 38
   top-predictor picks are one of these instead of real signal.
2. **Cell-type stratification on the model's actual pick.** nb11's
   nonlinearity check applied `stratified_linear_check` to the *cognate*
   gene. Here we apply it to the *model's actual top-predictor gene* for
   the 38 unexplained pairs -- checking whether the correlation the model
   found is a real gene-level relationship, or is itself explained by cell
   type (the same confound story this project keeps surfacing, just applied
   to a different pair).

### Extended technical-gene re-check

In [ ]:
from src.analysis.pathway_validation import flag_technical_genes, EXTENDED_TECHNICAL_GENE_PATTERNS

unexplained = pd.read_csv(RESULTS_DIR / f'{MODEL_VARIANT}_unexplained_pairs.csv')
print(f'Unexplained pairs to re-check: {len(unexplained)}')

unexplained['top_predictor_is_technical_extended'] = flag_technical_genes(
    unexplained['top_predictor_gene'], EXTENDED_TECHNICAL_GENE_PATTERNS,
)
newly_caught = unexplained[unexplained['top_predictor_is_technical_extended']]
print(f'Newly caught by the extended pattern list: {len(newly_caught)} / {len(unexplained)}')
newly_caught[['protein', 'cognate_gene', 'top_predictor_gene']]

### Load RNA/protein data
Scoped to just the genes/proteins appearing in the unexplained set (small --
no need to rebuild the full ~2000-gene union). Same normalization pipeline
as nb11 (log1p CP10k RNA, CLR protein, library-size regression on both).

In [ ]:
import scanpy as sc
import numpy as np

CHECKPOINT_PATH = BASE_PATH / 'data' / 'processed' / 'covid_subsampled.h5ad'
CELL_TYPE_COLUMN = 'cell_type'  # match nb11 -- adjust if the actual column differs

needed_genes = pd.unique(unexplained[['cognate_gene', 'top_predictor_gene']].values.ravel()).tolist()
needed_proteins = unexplained['protein'].unique().tolist()

covid = sc.read_h5ad(CHECKPOINT_PATH)
gex_mask = covid.var['feature_types'] == 'Gene Expression'
adt_mask = covid.var['feature_types'] == 'Antibody Capture'
covid_gex = covid[:, gex_mask].copy()
covid_adt = covid[:, adt_mask].copy()

genes_present = [g for g in needed_genes if g in covid_gex.var_names]
missing_genes = set(needed_genes) - set(genes_present)
if missing_genes:
    print(f'Warning: {len(missing_genes)} genes not found, dropping: {missing_genes}')

proteins_present = [p for p in needed_proteins if p in covid_adt.var_names]


def normalize_rna(adata_gex: sc.AnnData, gene_list: list[str]) -> sc.AnnData:
    """Log1p(CP10k) normalization on raw counts, restricted to gene_list. Same as nb11."""
    adata = adata_gex[:, gene_list].copy()
    adata.X = adata.layers['raw'].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    return adata


def clr_normalize(counts: np.ndarray) -> np.ndarray:
    """Centered log-ratio normalization per cell. Same as nb11."""
    log_counts = np.log1p(counts)
    return log_counts - log_counts.mean(axis=1, keepdims=True)


def regress_out_library_size(X: np.ndarray, raw_counts_layer: np.ndarray) -> np.ndarray:
    """Residualize each column of X against log1p(per-cell total raw counts). Same as nb11."""
    lib_size = np.log1p(np.asarray(raw_counts_layer).sum(axis=1)).reshape(-1, 1)
    design = np.column_stack([np.ones(X.shape[0]), lib_size])
    beta, _, _, _ = np.linalg.lstsq(design, X, rcond=None)
    return (X - design @ beta).astype(np.float32)


rna_adata = normalize_rna(covid_gex, genes_present)
X_check = np.asarray(rna_adata.X.todense()) if hasattr(rna_adata.X, 'todense') else np.asarray(rna_adata.X)
X_check = X_check.astype(np.float32)
rna_raw = rna_adata.layers['raw']
rna_raw = np.asarray(rna_raw.todense()) if hasattr(rna_raw, 'todense') else np.asarray(rna_raw)
X_check = regress_out_library_size(X_check, rna_raw)

adt_counts = covid_adt[:, proteins_present].layers['raw']
adt_counts = np.asarray(adt_counts.todense()) if hasattr(adt_counts, 'todense') else np.asarray(adt_counts)
Y_check = regress_out_library_size(clr_normalize(adt_counts).astype(np.float32), adt_counts)

cell_type_labels = covid_gex.obs[CELL_TYPE_COLUMN].values
print(f'X_check: {X_check.shape}, Y_check: {Y_check.shape}')

### Stratified check on the model's actual top-predictor gene
Note the column rename below: `nonlinearity_check`/`stratified_linear_check`
expect a `cognate_gene` column, but here it holds the model's **top predictor
gene**, not the true cognate gene -- we're asking the same question
(pooled-line vs. per-cell-type-lines vs. flexible fit) about a different pair.

In [ ]:
from src.analysis.evaluate import nonlinearity_check, stratified_linear_check, classify_relationship_source

top_pred_pairs = unexplained[
    unexplained['top_predictor_gene'].isin(genes_present) & unexplained['protein'].isin(proteins_present)
][['protein', 'top_predictor_gene']].rename(columns={'top_predictor_gene': 'cognate_gene'})

nonlin_unexplained = nonlinearity_check(X_check, Y_check, genes_present, proteins_present, top_pred_pairs)
strat_unexplained = stratified_linear_check(X_check, Y_check, genes_present, proteins_present, top_pred_pairs, cell_type_labels)

unexplained_nonlin = nonlin_unexplained.merge(strat_unexplained, on=['protein', 'cognate_gene'])
unexplained_nonlin = unexplained_nonlin.rename(columns={'cognate_gene': 'top_predictor_gene'})
unexplained_nonlin['confound_explained_gap'] = unexplained_nonlin['stratified_linear_r2'] - unexplained_nonlin['linear_r2']
unexplained_nonlin['residual_nonlinearity_gap'] = unexplained_nonlin['binned_r2'] - unexplained_nonlin['stratified_linear_r2']
unexplained_nonlin['category'] = classify_relationship_source(unexplained_nonlin)

print(unexplained_nonlin['category'].value_counts())
print()
print(f"cell_type_driven: the apparent top-predictor relationship depends heavily on cell-type grouping")
print(f"consistent_across_cell_types: a real relationship, not primarily a cell-type artifact")
print(f"weak_or_no_relationship: no meaningful relationship even accounting for cell type")

unexplained_nonlin.sort_values('category').reset_index(drop=True)

### Save follow-up results

In [ ]:
unexplained_followup = unexplained.merge(
    unexplained_nonlin[['protein', 'top_predictor_gene', 'linear_r2', 'stratified_linear_r2',
                         'binned_r2', 'confound_explained_gap', 'residual_nonlinearity_gap', 'category']],
    on=['protein', 'top_predictor_gene'], how='left',
)
unexplained_followup.to_csv(RESULTS_DIR / f'{MODEL_VARIANT}_unexplained_followup.csv', index=False)

print(f"Saved to {RESULTS_DIR / f'{MODEL_VARIANT}_unexplained_followup.csv'}")
print(f"  Extended-technical-flagged: {unexplained['top_predictor_is_technical_extended'].sum()} / {len(unexplained)}")
print(f"  cell_type_driven: {(unexplained_nonlin['category'] == 'cell_type_driven').sum()}")
print(f"  consistent_across_cell_types: {(unexplained_nonlin['category'] == 'consistent_across_cell_types').sum()}")
print(f"  weak_or_no_relationship: {(unexplained_nonlin['category'] == 'weak_or_no_relationship').sum()}")

## Summary
Fill in after running:
- Trustworthy core size (cognate rank-1 AND bootstrap-stable):
- Artifact-flagged size:
- Query set (gray-zone) size:
- Biologically supported fraction of the gray zone:
- Notable regulatory (TRRUST) or complex hits worth highlighting in the pitch:
- Unexplained pairs worth a manual literature check:
- How many of the 38 unexplained pairs are newly caught by the extended technical-gene list?
- How many of the remaining unexplained pairs are cell_type_driven vs. consistent_across_cell_types?
- Revised defensible headline number after this follow-up: